In [ ]:
%run PIKE.ipynb

In [ ]:
#the non-sparse setting with K = p.


import numpy as np
import pandas as pd
import multiprocessing as mp
from pathos.multiprocessing import ProcessingPool as Pool
import time
import os
import traceback



significance_level=0.05
kernelx='np';kernely='np'
r=0.7
la_candidates = 2**np.linspace(5,-10,15)

def example1(n,p,d):
    data_Z=np.reshape(np.random.uniform(0,2*np.pi,n*p),(n,p))
    data_X=np.reshape(np.random.uniform(0,2*np.pi,n*p),(n,p))
    data_Y=np.reshape(np.random.uniform(0,2*np.pi,n*p),(n,p))
    if d>0:
        A=np.reshape(np.random.binomial(1, 0.6,n*p),(n,p))
        index=int(d*0.4)
        data_Y[:,0:index]=(A*data_X)[:,0:index]*1
        data_X[:,index:d]=1*np.sin(data_Z[:,index:d])
        data_Y[:,index:d]=1*np.cos(data_Z[:,index:d])**2
    return data_X,data_Y
    

def process_single_iteration(args):

    i, n, p, d = args
    
    # Set a different random seed for each task
    seed = i
    np.random.seed(seed)
    
    # Generate data
    data_X, data_Y = example1(n, p, d)
        
    atau1,atau2,atau3 = com_PIKE(data_X, data_Y, kernelx, kernely, significance_level,r,la_candidates)
        
        
    # 返回结果
    return [atau3]  

    
def process_batch_parallel(batch_indices, n, p, d, batch_num, total_batches, pool):
    """Process a batch of tasks using process pool"""
    print(f"\nStarting batch {batch_num+1}/{total_batches}, containing {len(batch_indices)} tasks")
    batch_start_time = time.time()
    
    # Prepare arguments
    args_list = [(i, n, p, d) for i in batch_indices]
    
    # Parallel execution (using pool.map)
    batch_results = pool.map(process_single_iteration, args_list)
    
    batch_time = time.time() - batch_start_time
    print(f"Batch {batch_num+1}/{total_batches} completed, time: {batch_time:.2f} seconds")
    print(np.mean(batch_results,0))
    return batch_results



# ========== Process initialization function ==========
def worker_init():
    """Run when each process initializes"""
    import os

    # Set Numba environment
    os.environ['NUMBA_THREADING_LAYER'] = 'workqueue'

    print(f"Process {os.getpid()} initialized")


if __name__ == "__main__":
    for p in [200,400,800,1600,3200]:
        n = 100
        for d in [p]:
            total_iterations = 300

            def example1(n,p,d):
                data_Z=np.reshape(np.random.uniform(0,2*np.pi,n*p),(n,p))
                data_X=np.reshape(np.random.uniform(0,2*np.pi,n*p),(n,p))
                data_Y=np.reshape(np.random.uniform(0,2*np.pi,n*p),(n,p))
                if d>0:
                    A=np.reshape(np.random.binomial(1, 0.6,n*p),(n,p))
                    index=int(d*0.4)
                    data_Y[:,0:index]=(A*data_X)[:,0:index]*1
                    data_X[:,index:d]=1*np.sin(data_Z[:,index:d])
                    data_Y[:,index:d]=1*np.cos(data_Z[:,index:d])**2
                return data_X,data_Y

    
            # Set number of processes (adjust based on CPU cores)
            n_jobs = min(mp.cpu_count(), 8)  # Use at most 8 processes to avoid R memory pressure
            batch_size = n_jobs * 2  # Each batch processes 2x the number of processes tasks
    
            # Create batch indices
            indices = list(range(total_iterations))
            batches = [indices[i:i + batch_size] for i in range(0, total_iterations, batch_size)]
            total_batches = len(batches)
    
            print(f"Total tasks: {total_iterations}")
            print(f"Number of processes used: {n_jobs}")
            print(f"Batch size: {batch_size}, total batches: {total_batches}")

            # Initialize results dict (adjust keys based on your original code)
            results_dict = {
                'r11': []
                # Add other keys as needed
            }

            try:
                # Try to use forkserver (faster than spawn and more friendly to Numba)
                mp.set_start_method('forkserver', force=True)
                print("Using forkserver start method")
            except RuntimeError:
                # If forkserver not available, use spawn
                mp.set_start_method('spawn', force=True)
                print("Using spawn start method")
    
            all_results = []
            start_time = time.time()
    
            # Create process pool (with initialization function)
            with Pool(processes=n_jobs) as pool:
                for batch_num, batch_indices in enumerate(batches):
                    try:
                        batch_results = process_batch_parallel(
                            batch_indices, n, p, d, batch_num, total_batches, pool
                        )
                        all_results.extend(batch_results)
                
                        # Output progress
                        completed = len(all_results)
                        elapsed = time.time() - start_time
                        print(f"Completed {completed}/{total_iterations} tasks, total time: {elapsed:.2f} seconds")
                
                           
                    except Exception as e:
                        print(f"Batch {batch_num+1} processing failed: {str(e)}")
                        continue
    
            # Fill results into dict
            for result in all_results:
                for idx, key in enumerate(results_dict.keys()):
                    if idx < len(result):
                        results_dict[key].append(result[idx])
    
            # Compute averages
            eg1_5 = [results_dict[key] for key in results_dict.keys()]
    
            total_time = time.time() - start_time
            print(f"\nAll tasks completed! Total time: {total_time:.2f} seconds")
            print(f"Average time per task: {total_time/total_iterations:.2f} seconds")
            print(f"Final results: {eg1_5}")
            
            # Save final results
            power = pd.DataFrame(np.mean(np.array([eg1_5[i] for i in range(1)]).T,0))
            filename = f'/n={n}_p={p}_d={d}_pic1s.csv'
            power.to_csv(filename)
            print(f"Results saved to: {filename}")
            
            
            


